# Deploy Verification (M2M OAuth)

Automated post-deploy checks for the Semantic Ground Truth app using the M2M
service principal (OAuth client-credentials flow). A 302 redirect means
missing `CAN_USE` or a non-OAuth token.

1. `GET /health` → 200
2. `GET /api/me` → 200, roles include `admin`
3. `GET /api/campaigns` → 200
4. `POST /mcp` initialize → 200

All checks must return 200. Any non-200 is a failure.

In [0]:
%pip install --upgrade databricks-sdk requests
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("app_url", "", "App URL (https://...)")
dbutils.widgets.text("app_name", "dev-semantic-ground-truth", "App name")
dbutils.widgets.text("scope_name", "semantic_ground_truth_credentials", "Secret scope")
dbutils.widgets.text("client_id_key", "m2m_client_id_dev", "M2M client_id key")
dbutils.widgets.text("client_secret_key", "m2m_client_secret_dev", "M2M client_secret key")
dbutils.widgets.text("catalog", "hls_fde_dev", "UC catalog")
dbutils.widgets.text("schema_name", "dev_matthew_giglia_semantic_ground_truth", "UC schema")
dbutils.widgets.text("mcp_service_name", "", "MCP service full name")

app_url = dbutils.widgets.get("app_url").rstrip("/")
app_name = dbutils.widgets.get("app_name")
scope_name = dbutils.widgets.get("scope_name")
client_id_key = dbutils.widgets.get("client_id_key")
client_secret_key = dbutils.widgets.get("client_secret_key")
catalog = dbutils.widgets.get("catalog")
schema_name = dbutils.widgets.get("schema_name")
mcp_service_name = dbutils.widgets.get("mcp_service_name")

# Auto-discover app_url from app name when not provided
if not app_url and app_name:
    from databricks.sdk import WorkspaceClient as _WC
    _app = _WC().apps.get(app_name)
    _raw_url = _app.url or ""
    app_url = _raw_url if _raw_url.startswith("https://") else f"https://{_raw_url}"
    print(f"Auto-discovered app_url from '{app_name}': {app_url}")

# Default mcp_service_name from catalog + schema
if not mcp_service_name and catalog and schema_name:
    mcp_service_name = f"{catalog}.{schema_name}.semantic-ground-truth-mcp"
    print(f"Derived mcp_service_name: {mcp_service_name}")

assert app_url, "app_url could not be resolved — provide app_url or app_name"
assert scope_name, "scope_name is required"
assert client_id_key, "client_id_key is required"
assert client_secret_key, "client_secret_key is required"

print(f"App URL:          {app_url}")
print(f"Scope:            {scope_name}")
print(f"Client ID key:    {client_id_key}")
print(f"Client secret key: {client_secret_key}")

In [0]:
import os
import requests as req
from databricks.sdk import WorkspaceClient

# Read M2M SPN credentials from the secret scope
m2m_client_id = dbutils.secrets.get(scope=scope_name, key=client_id_key)
m2m_client_secret = dbutils.secrets.get(scope=scope_name, key=client_secret_key)

# Build an M2M OAuth WorkspaceClient (client-credentials flow)
host = os.environ.get("DATABRICKS_HOST", "https://fevm-hls-fde.cloud.databricks.com")
m2m_w = WorkspaceClient(
    host=host,
    client_id=m2m_client_id,
    client_secret=m2m_client_secret,
)

auth_headers = m2m_w.config.authenticate()
session = req.Session()
session.headers.update(auth_headers)
# Never follow redirects — a 302 means missing CAN_USE or wrong token type
session.max_redirects = 0

print(f"Auth type: {m2m_w.config.auth_type}")
print(f"Auth headers obtained: {list(auth_headers.keys())}")

# Sanity check
_test = session.get(f"{app_url}/health", timeout=30, allow_redirects=False)
print(f"Auth sanity: GET /health -> {_test.status_code}")
if _test.status_code == 302:
    raise RuntimeError("302 redirect — M2M SPN lacks CAN_USE on the app")

In [0]:
import json

results = []

def check(name: str, passed: bool, detail: str = ""):
    results.append({"check": name, "result": "PASS" if passed else "FAIL", "detail": detail})
    status = "\u2713" if passed else "\u2717"
    print(f"  {status} {name}: {detail}")

print("=== Verification Checks (M2M OAuth) ===")
print()

# --- Check 1: GET /health ---
print("[1] Health endpoint")
try:
    r = session.get(f"{app_url}/health", timeout=30, allow_redirects=False)
    check("GET /health", r.status_code == 200, f"status={r.status_code}")
except Exception as e:
    check("GET /health", False, str(e))

# --- Check 2: GET /api/me (roles include admin) ---
print("[2] API identity + admin role")
try:
    r = session.get(f"{app_url}/api/me", timeout=30, allow_redirects=False)
    check("GET /api/me status", r.status_code == 200, f"status={r.status_code}")
    if r.status_code == 200:
        body = r.json()
        roles = body.get("roles", [])
        check("admin role present", "admin" in roles, f"roles={roles}")
    else:
        check("admin role present", False, f"Cannot check — /api/me returned {r.status_code}")
except Exception as e:
    check("GET /api/me", False, str(e))

# --- Check 3: GET /api/campaigns ---
print("[3] Campaigns endpoint")
try:
    r = session.get(f"{app_url}/api/campaigns", timeout=30, allow_redirects=False)
    check("GET /api/campaigns", r.status_code == 200, f"status={r.status_code}")
except Exception as e:
    check("GET /api/campaigns", False, str(e))

# --- Check 4: POST /mcp initialize ---
print("[4] MCP initialize")
try:
    mcp_payload = {
        "jsonrpc": "2.0", "id": 1, "method": "initialize",
        "params": {
            "protocolVersion": "2025-03-26",
            "capabilities": {},
            "clientInfo": {"name": "deploy-verification", "version": "1.0.0"},
        },
    }
    r = session.post(
        f"{app_url}/mcp", json=mcp_payload,
        headers={"Accept": "application/json, text/event-stream"},
        timeout=30, allow_redirects=False,
    )
    check("POST /mcp initialize", r.status_code == 200, f"status={r.status_code}")
    if r.status_code == 200:
        # Streamable HTTP MCP answers with JSON or an SSE stream (`data: {...}` lines)
        text = (r.text or "").strip()
        if text.startswith("{"):
            body = json.loads(text)
        else:
            data_lines = [ln[5:].strip() for ln in text.splitlines() if ln.startswith("data:")]
            body = next((json.loads(d) for d in data_lines if d.startswith("{")), {})
        server_name = (body.get("result") or {}).get("serverInfo", {}).get("name")
        check("MCP server info", bool(server_name), f"server={server_name}")
except Exception as e:
    check("MCP server info", False, f"{type(e).__name__}: {e}")

# --- Summary ---
print()
print("=== Summary ===")
passed = sum(1 for r in results if r["result"] == "PASS")
failed = sum(1 for r in results if r["result"] == "FAIL")
print(f"Passed: {passed}, Failed: {failed}, Total: {len(results)}")
print()
for r in results:
    print(f"  [{r['result']}] {r['check']}: {r['detail']}")

output = json.dumps({"passed": passed, "failed": failed, "success": failed == 0, "checks": results})
if failed > 0:
    # Raise (not exit) so the job run is marked FAILED; dbutils.notebook.exit would end it as SUCCESS
    raise RuntimeError(f"Verification failed: {failed} check(s) did not pass: {output}")

dbutils.notebook.exit(output)